In [1]:
import sys
sys.path.append(r"C:\Users\zhaoz\Desktop\Research\Cooperation\coop_ephys_analysis")

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from external.diff_fam_social_memory_ephys.trodes import read_exported as tre

path = r"C:\Users\zhaoz\Desktop\Research\Cooperation\ecu_behaviors"

In [2]:
def pickle_this(thing_to_pickle, file_name):
    """
    Pickles things
    Args (2):   
        thing_to_pickle: anything you want to pickle
        file_name: str, filename that ends with .pkl 
    Returns:
        none
    """
    with open(file_name,'wb') as file:
        pickle.dump(thing_to_pickle, file)

def unpickle_this(pickle_file):
    """
    Unpickles things
    Args (1):   
        file_name: str, pickle filename that already exists and ends with .pkl
    Returns:
        pickled item
    """
    with open(pickle_file, 'rb') as file:
        return(pickle.load(file))

# Behavior Extraction

## Stage 4 Only

In [3]:
box_to_ecu_dict = {
            1: {'dio_ECU_Din1': 'selfish light',
                'dio_ECU_Din2': 'coop light',
                'dio_ECU_Din6': 'selfish nose poke',
                'dio_ECU_Din10': 'coop nose poke',
                'dio_ECU_Din8': 'subject port entry',
                'dio_ECU_Din16': 'recipient port entry'},   
            2: {'dio_ECU_Din3': 'selfish light',
                'dio_ECU_Din4': 'coop light',
                'dio_ECU_Din22': 'selfish nose poke',
                'dio_ECU_Din26': 'coop nose poke',
                'dio_ECU_Din24': 'subject port entry',
                'dio_ECU_Din32': 'recipient port entry'}}

# Read the box distributions for different recordings
df = pd.read_csv(path + "\Coop_ephys_Box_organization(Sheet1).csv")
df["Individual name"] = df["Individual name"].astype(str) + "_merged"
print(df.to_string)

<bound method DataFrame.to_string of                              Individual name         Trainning  Box  \
0   20250517_115803_alternates_D1_1-2_merged  Stage 3 last day  1.0   
1   20250517_115803_alternates_D1_1-3_merged  Stage 3 last day  1.0   
2   20250517_115803_alternates_D1_2-4_merged  Stage 3 last day  2.0   
3   20250517_115803_alternates_D1_2-1_merged  Stage 3 last day  2.0   
4   20250507_103358_alternates_D1_6-1_merged  Stage 3 last day  1.0   
..                                       ...               ...  ...   
93      20250517_134054_Stage4_D8_4-2_merged      Stage4 day 8  2.0   
94      20250517_134054_Stage4_D8_4-3_merged      Stage4 day 8  1.0   
95      20250517_134054_Stage4_D8_4-1_merged      Stage4 day 8  1.0   
96      20250517_121746_Stage4_D8_6-1_merged      Stage4 day 8  1.0   
97      20250517_121746_Stage4_D8_6-3_merged      Stage4 day 8  2.0   

   Mice condition  
0         Subject  
1       Recipient  
2         Subject  
3       Recipient  
4         

## Useful Functions

In [18]:
def trodes_behaviors_to_timestamps(name, data, clockrate, first_timestamp):
	'''
	Convert the data in .dat file format to normal behavior numpy array
	'''
	behavior_array = []
	event = []
	status = 1

	for (time, value) in data[1:]:
		if value != status and not name == "20250516_111920_Stage4_D7_2-1_merged" and not name == "20250516_111920_Stage4_D7_2-4_merged":
			raise ValueError('Invalid timestamp - the event has not started/stopped' + ' - ' + name)
		
		event.append((time - int(first_timestamp))/int(clockrate)) #align behavior data to 0
		if (status == 0):
			behavior_array.append(event)
			event = []
		status = 1 - status
	
	return np.array(behavior_array, dtype=float).tolist()

In [23]:
def load_trodes_recording(absolute_path):
	'''
	Read the relative path to the folder that contains merged.DIO folder
	'''
	name = Path(absolute_path).stem #20250508_100203_Stage4_D1_1-2_merged
	box = int(df.loc[df["Individual name"] == name, "Box"].iloc[0])	#1 or 2
	behaviors_dict = {}

	# Walk through the .dat files
	for root, dirs, files in os.walk(absolute_path):
		for file in files:
			if file.endswith('.dat'):
				din = file.split('.')[1]
				if din in box_to_ecu_dict[box]:	#if the din channel is meaningful
					data = tre.read_trodes_extracted_data_file(os.path.join(absolute_path, file))
					event = box_to_ecu_dict[box][din]
					behavior_data = data['data']
					# print(event, len(behavior_data))
					# print(data['first_timestamp'])
					# print(behavior_data[:10])
					behavior_array = trodes_behaviors_to_timestamps(name, behavior_data, data['clockrate'], data['first_timestamp'])
					behaviors_dict[event] = behavior_array

	return (name, behaviors_dict)

In [24]:
load_trodes_recording(r"C:\Users\zhaoz\Desktop\Research\Cooperation\ecu_behaviors\Stage4_D1\20250508_100203_Stage4_D1_1-2_merged.DIO")


('20250508_100203_Stage4_D1_1-2_merged',
 {'selfish light': [[47.44435, 52.4442],
   [54.48395, 59.48405],
   [69.57465, 74.57435],
   [76.06425, 81.0643],
   [83.27445, 88.2744],
   [92.2647, 97.2645],
   [100.65455, 105.6546],
   [110.05475, 115.05495],
   [117.51475, 122.5149],
   [126.27485, 131.2749],
   [135.25535, 140.25505],
   [147.7151, 152.7152],
   [160.91525, 165.9155],
   [169.0854, 174.0857],
   [177.2555, 182.2557],
   [187.546, 192.54555],
   [196.2057, 201.2062],
   [204.58595, 209.5859],
   [212.02595, 217.026],
   [222.44605, 227.44635],
   [234.1962, 239.19625],
   [243.08655, 248.08635],
   [250.3164, 255.31635],
   [257.8965, 262.89655],
   [266.6166, 271.6169],
   [273.98685, 278.98675],
   [282.0568, 287.05685],
   [289.647, 294.64695],
   [302.68705, 307.6875],
   [309.95715, 314.95735],
   [318.14725, 323.1474],
   [326.23735, 331.2374],
   [353.9477, 358.94775],
   [361.4879, 366.48785],
   [375.11845, 380.118],
   [397.8082, 402.8084],
   [403.9483, 408.948

In [25]:
def print_n_o_events(behaviors_dict):
	for key, value in behaviors_dict.items():
		print('#', key, "-", len(value))

### Extract from stage 4 folder

In [26]:
# Walk through directories of each day in stage 4
for root, dirs, files in os.walk(path):
	for daily_directory in dirs:
		if daily_directory.startswith("Stage4"):
			daily_path = os.path.join(path, daily_directory)
			daily_behavior_dict = {}
			daily_count = 0
			#walk through every DIO channels
			for root, dirs, files in os.walk(daily_path):
				for dio_directory in dirs:
					if dio_directory.endswith("DIO"):
						# print(dio_directory)
						dio_path = os.path.join(daily_path, dio_directory)
						rec_name, behaviors = load_trodes_recording(dio_path)
						daily_behavior_dict[rec_name] = behaviors
						daily_count += 1

			print("In " + daily_directory + ", " + str(daily_count) + " mice have been processed.")
			file_name = daily_directory + ".pkl"
			pickle_this(daily_behavior_dict, os.path.join(path, file_name))

In Stage4_D1, 10 mice have been processed.
In Stage4_D2, 7 mice have been processed.
In Stage4_D3, 10 mice have been processed.
In Stage4_D4, 10 mice have been processed.
In Stage4_D5, 10 mice have been processed.
In Stage4_D6, 9 mice have been processed.
In Stage4_D7, 10 mice have been processed.
In Stage4_D8, 10 mice have been processed.


### Unpickle to validate

In [27]:
Stage4_D1_behaviors = unpickle_this(path + "\Stage4_D1.pkl")
print(Stage4_D1_behaviors.keys())

dict_keys(['20250508_100203_Stage4_D1_1-2_merged', '20250508_100203_Stage4_D1_1-3_merged', '20250508_100203_Stage4_D1_2-1_merged', '20250508_100203_Stage4_D1_2-4_merged', '20250508_112121_Stage4_D1_6-1_merged', '20250508_112121_Stage4_D1_6-3_merged', '20250509_131426_Stage4_D1_4-1_merged', '20250509_131426_Stage4_D1_4-2_merged', '20250509_131426_Stage4_D1_4-3_merged', '20250509_131426_Stage4_D1_4-4_merged'])


## Jitters Check - Migrated

In [ ]:
def events_duration_distribution(behaviors_dict, behavior, xmin=0, xmax=1, mode='multiple recordings'):
	'''
	Plot a histogram of event durations for a given behavior

	Args (2):
		behaviors_dict: dictionary, a nested dictionary with the structure:
			{
				"recording_id": {
					"behavior_name": np.ndarray of shape (n, 2),
					...
				},
				...
			}
		behavior: str, the single behavior that will be plotted
	'''
	if mode == 'single recording': 
		behavior_array = np.array(behaviors_dict[behavior])
	elif mode == 'multiple recordings':
		behavior_array = []
		for rec, rec_behaviors in behaviors_dict.items():
			if len(rec_behaviors[behavior]) > 0:
				for event in rec_behaviors[behavior]:
					behavior_array.append(event)
		behavior_array = np.array(behavior_array)
	
	durations = behavior_array[:, 1] - behavior_array[:, 0]
	
	# Plotting
	plt.figure(figsize=(12, 5))
	plt.hist(durations, bins=30, range=(xmin, xmax), color='#15616F', edgecolor="black")
	plt.title("Distribution of Event Durations - " + mode, fontsize=16)
	plt.xlabel(behavior + " event durations (seconds)", fontsize=14)
	plt.ylabel("# of Events", fontsize=14)
	ticks = np.linspace(xmin, xmax, 10)
	plt.xticks(np.round(ticks, 2))
	plt.show()

def between_events_duration_distribution(behaviors_dict, behaviors, xmin=0, xmax=1, mode='multiple recordings'):
	'''
	Plot a histogram of durations between separate events (can be the same or different behaviors)

	Args (2):
		behaviors_dict: dictionary, a nested dictionary with the structure:
			{
				"recording_id": {
					"behavior_name": np.ndarray of shape (n, 2),
					...
				},
				...
			}
		behaviors: list, a list of behaviors that will be plotted
	'''
	behavior_array = []
	if mode == 'single recording':
		for behavior in behaviors:
			for event in behaviors_dict[behavior]:
				behavior_array.append(event)
		behavior_array.sort(key=lambda x: x[0])
		total_between_event_durations = np.array(behavior_array)[1:, 0] - np.array(behavior_array)[:-1, 1]
		
	elif mode == 'multiple recordings':
		total_between_event_durations = np.array([])
		for rec, rec_behaviors in behaviors_dict.items():
			behavior_array = []
			for behavior in behaviors:
				for event in rec_behaviors[behavior]:
					behavior_array.append(event)
			behavior_array.sort(key=lambda x: x[0])
			rec_between_event_durations = np.array(behavior_array)[1:, 0] - np.array(behavior_array)[:-1, 1]
			total_between_event_durations = np.concatenate((total_between_event_durations, rec_between_event_durations))
	
	print(total_between_event_durations.size)

	plt.figure(figsize=(12, 5))
	plt.hist(total_between_event_durations, bins=30, range=(xmin, xmax), color='#FFAF00', edgecolor="black")
	plt.title("Distribution of Between-event Durations - " + mode, fontsize=16)
	plt.xlabel(f"Time Gaps Between {' & '.join(behaviors)}", fontsize=14)
	plt.ylabel("# of Events", fontsize=14)
	ticks = np.linspace(xmin, xmax, 10)
	plt.xticks(np.round(ticks, 2))
	plt.show()

In [ ]:
events_duration_distribution(Stage4_D1_behaviors['20250508_100203_Stage4_D1_1-2_merged'], "coop nose poke", mode="single recording")

In [ ]:
events_duration_distribution(Stage4_D1_behaviors, "coop nose poke")

In [ ]:
# One recording
between_events_duration_distribution(Stage4_D1_behaviors['20250508_100203_Stage4_D1_1-2_merged'], ["coop nose poke", "subject port entry"], mode="single recording")

In [ ]:
# One day of multiple recordings
between_events_duration_distribution(Stage4_D1_behaviors, ["coop nose poke", "subject port entry"], mode="multiple recordings")